1: Load Environment & API Key

In [ ]:
import os
from dotenv import load_dotenv

# Membaca file .env yang sama
load_dotenv()

# Cek apakah API Key sudah terbaca dengan benar
if os.getenv("GOOGLE_API_KEY"):
    print("✅ API Key Gemini berhasil dimuat!")
else:
    print("❌ API Key BELUM terbaca. Pastikan file .env berada satu folder dengan file .ipynb ini.")

2: Load Dokumen PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Membaca file dokumen.pdf yang sudah kamu download tadi
loader = PyPDFLoader("Esai_Untuk_Indonesia.pdf")
docs = loader.load()

print(f"✅ Berhasil membaca PDF! Total halaman: {len(docs)}")

3: Potong Teks (Text Splitting)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(f"✅ Teks berhasil dipecah menjadi {len(splits)} bagian kecil.")

In [ ]:
!pip install sentence-transformers

4: Inisialisasi Embedding & Simpan ke Chroma DB

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embedding alternatif gratis yang berjalan lokal di laptop tanpa API Key Google
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Simpan ke Chroma
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print("✅ Vectorstore Chroma dan Embeddings sukses dibuat secara lokal!")

In [ ]:
!pip install langchain-community

In [ ]:
!pip install -U langchainhub

In [ ]:
!pip install langchain-openai

5: Jalankan RAG dengan Gemini AI (Output Akhir)

In [ ]:
import time
import os
import wandb
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.llms import HuggingFaceEndpoint
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ====================================================
# 1. KONFIGURASI AWAL & DATA INPUT
# ====================================================
# Nonaktifkan W&B sementara untuk testing cepat secara lokal
os.environ["WANDB_MODE"] = "disabled"
run = wandb.init(project="Generative-AI-RAG", name="RAG_6_Models_Benchmark")

# Pertanyaan Eksperimen (Dinaikkan ke atas agar terbaca oleh model di bawah)
query = "Apa topik utama yang dibahas dalam dokumen ini?"

# Inisialisasi awal variabel jawaban (Default jika model dilewati/eror)
jawaban_gn_murni = jawaban_gn_rag = "Limit Kuota Tercapai (429)"
jawaban_gc_murni = jawaban_gc_rag = "Limit Kuota Tercapai (429)"
jawaban_ds_murni = jawaban_ds_rag = "Gagal Memproses"
jawaban_gpt_murni = jawaban_gpt_rag = "Dilewati (Di-komentar)"
jawaban_pro_murni = jawaban_pro_rag = "Dilewati (Di-komentar)"
jawaban_gemma_murni = jawaban_gemma_rag = "Dilewati (Di-komentar)"

# --- PROMPT TEMPLATE ---
prompt_murni = ChatPromptTemplate.from_messages([("human", "{input}")])

system_prompt = (
    "Anda adalah asisten untuk tugas tanya jawab. "
    "Gunakan potongan konteks berikut untuk menjawab pertanyaan. "
    "Jika Anda tidak tahu jawabannya, katakan saja Anda tidak tahu.\n\n"
    "{context}"
)
prompt_rag = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# ====================================================
# 2. EKSEKUSI MODEL SECARA BERGANTIAN (ANTI-CRASH)
# ====================================================

print("⏳ Memulai komparasi model benchmark...")
print("-" * 50)

# --- A. GEMINI 2.5 FLASH (TEMP 0.3) ---
try:
    print("🤖 Memproses Gemini Temp 0.3...")
    llm_gemini_n = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

    # Jalankan Jawaban Murni
    chain_gn_murni = prompt_murni | llm_gemini_n | StrOutputParser()
    jawaban_gn_murni = chain_gn_murni.invoke({"input": query})

    # Jalankan Jawaban RAG
    rag_gn = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_gemini_n | StrOutputParser())
    jawaban_gn_rag = rag_gn.invoke(query)
    print("✅ Gemini Temp 0.3 Selesai.")
except Exception as e:
    print(f"⚠️ Gemini Temp 0.3 Dilewati: {e}")

# Beri Jeda 4 detik agar terhindar dari pembatasan limit Google API (RPM)
time.sleep(4)

# --- B. GEMINI 2.5 FLASH (TEMP 0.7) ---
try:
    print("🤖 Memproses Gemini Temp 0.7...")
    llm_gemini_c = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

    # Jalankan Jawaban Murni
    chain_gc_murni = prompt_murni | llm_gemini_c | StrOutputParser()
    jawaban_gc_murni = chain_gc_murni.invoke({"input": query})

    # Jalankan Jawaban RAG
    rag_gc = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_gemini_c | StrOutputParser())
    jawaban_gc_rag = rag_gc.invoke(query)
    print("✅ Gemini Temp 0.7 Selesai.")
except Exception as e:
    print(f"⚠️ Gemini Temp 0.7 Dilewati: {e}")

# --- C. DEEPSEEK-R1 (HUGGING FACE) ---
try:
    print("🤖 Memproses DeepSeek-R1...")
    llm_deepseek = HuggingFaceEndpoint(
        repo_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        task="text-generation",
        temperature=0.3
    )
    # Jalankan Jawaban Murni
    chain_ds_murni = prompt_murni | llm_deepseek | StrOutputParser()
    jawaban_ds_murni = chain_ds_murni.invoke({"input": query})

    # Jalankan Jawaban RAG
    rag_ds = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_deepseek | StrOutputParser())
    jawaban_ds_rag = rag_ds.invoke(query)
    print("✅ DeepSeek-R1 Selesai.")
except Exception as e:
    print(f"⚠️ DeepSeek Gagal: {e}")
    jawaban_ds_murni = jawaban_ds_rag = "Error DeepSeek"


# ====================================================
# 3. CETAK OUTPUT KONSOL NOTEBOOK (ESTETIK)
# ====================================================
print("\n" + "="*60)
print("🛑 HASIL BENCHMARK RAG MULTI-MODEL")
print("="*60)
print(f"❓ PERTANYAAN: '{query}'\n")

print("-" * 60)
print("🟢 1. GEMINI 2.5 FLASH (TEMP 0.3)")
print(f"   [MURNI] : {jawaban_gn_murni.strip()}")
print(f"   [RAG]   : {jawaban_gn_rag.strip()}")

print("-" * 60)
print("🔵 2. GEMINI 2.5 FLASH (TEMP 0.7)")
print(f"   [MURNI] : {jawaban_gc_murni.strip()}")
print(f"   [RAG]   : {jawaban_gc_rag.strip()}")

print("-" * 60)
print("🐋 3. DEEPSEEK-R1 (QWEN-7B)")
print(f"   [MURNI] : {jawaban_ds_murni.strip()}")
print(f"   [RAG]   : {jawaban_ds_rag.strip()}")

print("-" * 60)
print("⚫ 4. ChatGPT (GPT-5.4-mini)  -> STATUS: Dilewati (Di-komentar)")
print("💤 5. GEMINI 2.5 PRO          -> STATUS: Dilewati (Di-komentar)")
print("💎 6. GEMMA 2                 -> STATUS: Dilewati (Di-komentar)")
print("="*60)


# ====================================================
# 4. LOGGING STRUKTUR KE WEIGHTS & BIASES TABLE
# ====================================================
compare_table = wandb.Table(columns=["Model", "Jawaban Murni (Tanpa RAG)", "Jawaban dengan RAG"])
compare_table.add_data("Gemini 2.5 Flash (Temp 0.3)", jawaban_gn_murni, jawaban_gn_rag)
compare_table.add_data("Gemini 2.5 Flash (Temp 0.7)", jawaban_gc_murni, jawaban_gc_rag)
compare_table.add_data("ChatGPT (GPT-5.4-mini)", jawaban_gpt_murni, jawaban_gpt_rag)
compare_table.add_data("DeepSeek-R1 (Qwen)", jawaban_ds_murni, jawaban_ds_rag)
compare_table.add_data("Gemini 2.5 Pro (Idle)", jawaban_pro_murni, jawaban_pro_rag)
compare_table.add_data("Gemma 2 (Idle)", jawaban_gemma_murni, jawaban_gemma_rag)

wandb.log({"Tabel_Benchmark_6_Model_RAG": compare_table})
run.finish()